# scikit-fem Tutorial: Basic FEM for a Square Plate

This notebook is a **from-scratch to intermediate** guide to using **scikit-fem**
for simple Finite Element Method (FEM) analysis in pure Python.

We’ll focus on a **2D square plate** subject to different kinds of loads:

- Transverse pressure (out-of-plane load in a 2D model → equivalent to body/traction load)
- Axial load (tension/compression along one axis)
- “Radial” style load (loads varying with distance from a point on the plate)

We’ll keep meshes and problems **simple** (few elements, coarse grids) so they are
conceptually clear and quick to solve.

### Contents

- [1. What is scikit-fem?](#1-what-is-scikit-fem)
- [2. Installing and Importing scikit-fem](#2-installing-and-importing-scikit-fem)
- [3. Basic Workflow: Define Mesh, Problem, Material, and Solve](#3-basic-workflow-define-mesh-problem-material-and-solve)
- [4. Example 1: Square Plate with Uniform Transverse Load](#4-example-1-square-plate-with-uniform-transverse-load)
- [5. Example 2: Square Plate in Axial Tension](#5-example-2-square-plate-in-axial-tension)
- [6. Example 3: Square Plate with Radial-Type Load](#6-example-3-square-plate-with-radial-type-load)
- [7. Post-Processing: Displacements and Stresses](#7-post-processing-displacements-and-stresses)
- [8. Mesh Refinement and Convergence Hints](#8-mesh-refinement-and-convergence-hints)
- [9. Notes on 3D, Nonlinear Materials, and Dynamic Problems](#9-notes-on-3d-nonlinear-materials-and-dynamic-problems)
- [10. Best Practices and When to Use scikit-fem in Quant/Engineering Contexts](#10-best-practices-and-when-to-use-scikit-fem-in-quantengineering-contexts)
- [11. FEM Perspective on the Black–Scholes Equation](#11-fem-perspective-on-the-blackscholes-equation)

## 1. What is scikit-fem?

**scikit-fem** is a pure-Python finite element library built on NumPy/SciPy.
It focuses on clarity and composability rather than black-box magic, making it
ideal for learning FEM and building small to medium FEM codes directly in Python.

With scikit-fem you can:

- Build meshes (1D/2D/3D) or import them via `meshio`.
- Define finite element spaces (scalar or vector fields, different element types).
- Assemble stiffness/mass matrices and load vectors with concise model helpers.
- Apply Dirichlet/Neumann boundary conditions.
- Solve linear elasticity, Poisson, and more using standard SciPy solvers.

For a quant/engineering developer, scikit-fem is useful for:

- Quick structural/elasticity simulations in notebooks.
- Experimenting with FEM formulations before moving to larger frameworks.
- Integrating FEM pieces into broader Python data/ML/quant pipelines.

## 2. Installing and Importing scikit-fem

Installation (from a shell):

```bash
pip install scikit-fem[all]
```

The `[all]` extra pulls in optional dependencies like `meshio` and `matplotlib`
for mesh I/O and visualization.

In this notebook, we’ll assume scikit-fem is installed and import it as:

```python
import skfem
```

If you don’t have it installed, you can still read through the code and run the
cells that don’t depend on it.

In [ ]:
# 2.1 Try importing scikit-fem

try:
    import skfem
    print("scikit-fem version imported successfully:", getattr(skfem, "__version__", "(version attr not available)"))
except Exception as e:
    print("Could not import scikit-fem (or it raised on import).")
    print("Error:", repr(e))
    skfem = None  # allow rest of notebook to be read without crashing

## 3. Basic Workflow: Define Mesh, Problem, Material, and Solve

The typical **scikit-fem** workflow:

1. **Create a mesh**: e.g. a unit square discretized into triangles.
2. **Define a finite element** and a **basis** on that mesh (e.g. P2 triangles for 2D displacement).
3. **Use model helpers** to assemble stiffness and load matrices for a PDE.
4. **Apply boundary conditions** by eliminating Dirichlet DOFs with `condense`.
5. **Solve** the linear system with SciPy’s sparse solvers.

In code this often looks like:

```python
from skfem import MeshTri, ElementTriP2, Basis, asm, solve, condense
from skfem.models.elasticity import linear_elasticity, linear_elasticity_load

m = MeshTri.init_tensor(np.linspace(0., 1., nx + 1),
                        np.linspace(0., 1., ny + 1))
e = ElementTriP2()

basis = Basis(m, e, intorder=4)

K = asm(linear_elasticity(E, nu), basis)

# body force f(x) in R^2
f_vec = asm(linear_elasticity_load(basis, load=body_force))

# Dirichlet boundary dofs on left edge
left_facets = m.facets_satisfying(lambda x: x[0] == 0.0)
D = basis.get_dofs(facets=left_facets).all()
x, Kc, fc = solve(*condense(K, f_vec, D=D))

# x contains displacement dofs
```

Next, we’ll write concrete examples for a **square plate** under different loads.

## 4. Example 1: Square Plate with Uniform Transverse Load

We consider a **unit square plate** in 2D:

- Domain: \([0, 1] \times [0, 1]\).
- Boundary conditions:
  - Left edge (x = 0) fixed (zero displacement) – representing a clamped edge.
  - Other edges free.
- Load:
  - Uniform “transverse” load modeled as a constant **body force** in the y-direction.

Steps:

1. Generate a simple triangular mesh for the unit square.
2. Set up a 2D linear elasticity problem with a simple isotropic material.
3. Apply boundary conditions and body force.
4. Solve and inspect maximum displacement.

In [ ]:
import numpy as np

import skfem
from skfem import MeshTri, ElementTriP2, ElementVector, Basis, asm, solve, condense, LinearForm
from skfem.models.elasticity import linear_elasticity

# 1. Mesh: unit square [0,1]x[0,1] with nx*ny rectangles split into triangles
nx, ny = 16, 16
m = MeshTri.init_tensor(np.linspace(0., 1., nx + 1),
                        np.linspace(0., 1., ny + 1))

# 2. Finite element and basis: 2D *vector* displacement, quadratic triangles
e = ElementVector(ElementTriP2())      # vector-valued P2 element (dim=2)
basis = Basis(m, e, intorder=4)

# 3. Material parameters (plane strain/stress equivalent)
E = 210e9   # Young's modulus (Pa)
nu = 0.3    # Poisson's ratio

# 4. Stiffness matrix for linear elasticity
K = asm(linear_elasticity(E, nu), basis)

# 5. Uniform body force in -y direction (N/m^3 equivalent)
body_force = np.array([0.0, -1e5])

@LinearForm
def load(v, w):
    # v[0], v[1] are the x- and y-components at quadrature points
    return body_force[0] * v[0] + body_force[1] * v[1]

f = asm(load, basis)

# 6. Dirichlet BC: fix left edge x = 0 (u = 0 there)
left_facets = m.facets_satisfying(lambda x: x[0] == 0.0)
D = basis.get_dofs(facets=left_facets).all()

# 7. Solve condensed system (follow official pattern)
x = solve(*condense(K, f, D=D))

# 8. Post-process: displacement magnitude per node
u = basis.interpolate(x).value  # shape (2, n_nodes)
u_mag = np.linalg.norm(u, axis=0)

print("Number of nodes:", u.shape[1])
print("Max displacement (uniform load):", u_mag.max())

In [ ]:
import matplotlib.pyplot as plt
from skfem.visuals.matplotlib import plot, draw

# Assumes:
# - m (MeshTri)
# - basis (Basis with ElementVector(ElementTriP2()))
# - x (solution vector)
# - body_force (np.array([0.0, -1e5]))

# Use y-displacement component as scalar field for plotting
# x contains vector DOFs; for ElementVector(ElementTriP2()) the y-component
# is every second DOF starting at index 1.
uy_dofs = x[1::2]

fig, ax = plt.subplots(figsize=(5, 5))

# Original mesh
draw(m, ax=ax, color='0.7', linewidth=0.5)

# Plot y-displacement on the mesh
plot(basis, uy_dofs, shading='gouraud', colorbar=True, ax=ax)

# Load arrows (body force direction)
k = 8
px = m.p[0, ::k]
py = m.p[1, ::k]
fx = np.full_like(px, body_force[0])
fy = np.full_like(py, body_force[1])
arrow_scale = 1e-5
ax.quiver(px, py, fx * arrow_scale, fy * arrow_scale,
          color='red', angles='xy', scale_units='xy', scale=1)

ax.set_aspect('equal')
ax.set_title('Example 1: uniform load\n(gray = mesh, colors = y-displacement, red = load)')
plt.show()

## 5. Example 2: Square Plate in Axial Tension

Now we apply an **axial load** along the x-direction:

- Domain: same unit square.
- Boundary conditions:
  - Left edge (x = 0) fixed.
  - Right edge (x = 1) subject to a uniform traction in +x (tensile load).

This corresponds to a simple uniaxial tension test in 2D. We expect primarily
elongation in x and some contraction in y (due to Poisson’s effect).

In scikit-fem, a traction can be modeled as a Neumann term integrated over the
right-edge facets. For simplicity we sketch the pattern and reuse the same mesh
and material parameters as in Example 1.

In [ ]:
# 5.1 Axial tension example (runnable scikit-fem example)

import numpy as np

import skfem
from skfem import MeshTri, ElementTriP2, ElementVector, Basis, asm, solve, condense, LinearForm
from skfem.models.elasticity import linear_elasticity

# 1. Mesh: unit square [0,1]x[0,1]
nx, ny = 16, 16
m = MeshTri.init_tensor(np.linspace(0., 1., nx + 1),
                        np.linspace(0., 1., ny + 1))

# 2. Finite element and basis: 2D vector displacement
e = ElementVector(ElementTriP2())
basis = Basis(m, e, intorder=4)

# 3. Material parameters
E = 210e9
nu = 0.3

# 4. Stiffness matrix
K = asm(linear_elasticity(E, nu), basis)

# 5. Axial body force in +x direction (approximating axial tension)
body_force_axial = np.array([1e5, 0.0])

@LinearForm
def load_axial(v, w):
    return body_force_axial[0] * v[0] + body_force_axial[1] * v[1]

f_axial = asm(load_axial, basis)

# 6. Dirichlet BC: fix left edge x = 0
left_facets = m.facets_satisfying(lambda x: x[0] == 0.0)
D = basis.get_dofs(facets=left_facets).all()

# 7. Solve condensed system
x_axial = solve(*condense(K, f_axial, D=D))

# 8. Displacement magnitude
u_axial = basis.interpolate(x_axial).value
u_mag_axial = np.linalg.norm(u_axial, axis=0)

print("Number of nodes (axial):", u_axial.shape[1])
print("Max displacement (axial load):", u_mag_axial.max())

In [ ]:
import matplotlib.pyplot as plt
from skfem.visuals.matplotlib import plot, draw

# Assumes:
# - m, basis as in Example 2
# - x_axial (solution vector for axial load)
# - body_force_axial (np.array([1e5, 0.]))

u_axial = basis.interpolate(x_axial).value
u_mag_axial = np.linalg.norm(u_axial, axis=0)

scale = 1e-6

fig, ax = plt.subplots(figsize=(5, 5))

draw(m, ax=ax, color='0.7', linewidth=0.5)

from skfem import MeshTri
p_def_axial = m.p + scale * u_axial
m_def_axial = MeshTri(p_def_axial, m.t)
draw(m_def_axial, ax=ax, color='C1', linewidth=0.5, alpha=0.7)

plot(Basis(m_def_axial, basis.elem, intorder=4), u_mag_axial,
     shading='gouraud', colorbar=True, ax=ax)

# Axial load arrows: along +x on a vertical line near x=1
k = 8
mask = np.isclose(m.p[0], 1.0)
px = m.p[0, mask][::k]
py = m.p[1, mask][::k]
fx = np.full_like(px, body_force_axial[0])
fy = np.full_like(py, body_force_axial[1])
arrow_scale = 1e-5
ax.quiver(px, py, fx * arrow_scale, fy * arrow_scale, color='red', angles='xy', scale_units='xy', scale=1)

ax.set_aspect('equal')
ax.set_title('Example 2: axial load\n(gray = original, orange = deformed, red = load direction)')
plt.show()

## 6. Example 3: Square Plate with Radial-Type Load

To approximate a **radial load** in 2D, we can:

- Choose a **load center** (e.g. plate center at (0.5, 0.5)).
- Apply a body force whose magnitude/direction depend on the vector from the
  center to each point.

A simple model:

- Force at point \(x\) is \(f(x) = k \cdot (x - x_c)\), where \(x_c\) is the center
  and \(k\) is a scalar coefficient.

In scikit-fem this can be expressed by passing a Python function as the `load`
argument to `linear_elasticity_load`, which receives coordinates and returns
force vectors.

In [ ]:
# 6.1 Radial-type load example (runnable scikit-fem example)

import numpy as np

import skfem
from skfem import MeshTri, ElementTriP2, ElementVector, Basis, asm, solve, condense, LinearForm
from skfem.models.elasticity import linear_elasticity

# 1. Mesh
nx, ny = 16, 16
m = MeshTri.init_tensor(np.linspace(0., 1., nx + 1),
                        np.linspace(0., 1., ny + 1))

# 2. Vector-valued element and basis
e = ElementVector(ElementTriP2())
basis = Basis(m, e, intorder=4)

# 3. Material
E = 210e9
nu = 0.3
K = asm(linear_elasticity(E, nu), basis)

# 4. Radial body force around center c
c = np.array([0.5, 0.5])
k = 1e5

@LinearForm
def load_radial(v, w):
    x = w.x                 # shape (2, n_elem, n_qp)
    r = x - c[:, None, None]  # broadcast c to (2, 1, 1)
    fx = k * r[0]
    fy = k * r[1]
    return fx * v[0] + fy * v[1]

f_radial = asm(load_radial, basis)

# 5. Clamp left edge again for reference
left_facets = m.facets_satisfying(lambda x: x[0] == 0.0)
D = basis.get_dofs(facets=left_facets).all()

# 6. Solve
x_radial = solve(*condense(K, f_radial, D=D))

u_radial = basis.interpolate(x_radial).value
u_mag_radial = np.linalg.norm(u_radial, axis=0)

print("Number of nodes (radial):", u_radial.shape[1])
print("Max displacement (radial load):", u_mag_radial.max())

In [ ]:
import matplotlib.pyplot as plt
from skfem.visuals.matplotlib import plot, draw

# Assumes:
# - m, basis as in Example 3
# - x_radial (solution vector)
# - c (center, np.array([0.5, 0.5]))
# - k (scalar for radial load)

u_radial = basis.interpolate(x_radial).value
u_mag_radial = np.linalg.norm(u_radial, axis=0)

scale = 1e-6

fig, ax = plt.subplots(figsize=(5, 5))

draw(m, ax=ax, color='0.7', linewidth=0.5)

from skfem import MeshTri
p_def_radial = m.p + scale * u_radial
m_def_radial = MeshTri(p_def_radial, m.t)
draw(m_def_radial, ax=ax, color='C2', linewidth=0.5, alpha=0.7)

plot(Basis(m_def_radial, basis.elem, intorder=4), u_mag_radial,
     shading='gouraud', colorbar=True, ax=ax)

# Radial load arrows: arrows pointing radially outward from c at a coarse subset of nodes
k_sel = 16
px = m.p[0, ::k_sel]
py = m.p[1, ::k_sel]
dx = px - c[0]
dy = py - c[1]
# Normalize and scale
norm = np.hypot(dx, dy)
norm[norm == 0.0] = 1.0
dxn = dx / norm
dyn = dy / norm
arrow_scale = 0.1  # purely visual
ax.quiver(px, py, dxn * arrow_scale, dyn * arrow_scale, color='red', angles='xy', scale_units='xy', scale=1)

ax.set_aspect('equal')
ax.set_title('Example 3: radial load\n(gray = original, green = deformed, red = load direction)')
plt.show()

## 7. Post-Processing: Displacements and Stresses

After solving, typical quantities of interest:

- **Nodal displacements**, $u(x)$
- **Strains**, $\varepsilon$
- **Stresses**, $\sigma$

In scikit-fem, `basis.interpolate(x)` lets you evaluate the displacement field
at nodes or quadrature points. For stresses, you can use model helpers or
manually compute strain/stress tensors from displacement gradients.

Below is a sketch of computing a von Mises-like quantity given stress
components (conceptually similar to the PolyFEM example).

In [ ]:
# 7.1 Example: simple post-processing of displacement field (runnable)

import numpy as np

import skfem
from skfem import MeshTri, ElementTriP2, ElementVector, Basis, asm, solve, condense, LinearForm, Functional
from skfem.models.elasticity import linear_elasticity

# Reuse the uniform-load setup from Example 1
nx, ny = 16, 16
m = MeshTri.init_tensor(np.linspace(0., 1., nx + 1),
                        np.linspace(0., 1., ny + 1))

e = ElementVector(ElementTriP2())
basis = Basis(m, e, intorder=4)

E = 210e9
nu = 0.3
K = asm(linear_elasticity(E, nu), basis)

body_force = np.array([0.0, -1e5])

@LinearForm
def load(v, w):
    return body_force[0] * v[0] + body_force[1] * v[1]

f = asm(load, basis)

left_facets = m.facets_satisfying(lambda x: x[0] == 0.0)
D = basis.get_dofs(facets=left_facets).all()

x = solve(*condense(K, f, D=D))

u = basis.interpolate(x)

# Example functional: L2 norm of displacement over the domain
@Functional
def disp_energy(w):
    uh = w['uh']  # vector field at quadrature points
    return uh[0]**2 + uh[1]**2

energy = disp_energy.assemble(basis, uh=u)
L2_u = np.sqrt(energy)

print("L2 norm of displacement (uniform load case):", L2_u)

## 8. Mesh Refinement and Convergence Hints

For FEM problems, accuracy depends strongly on **mesh resolution** and **element
order**:

- Coarse meshes (few elements) are fine for conceptual demos but may give rough
  approximations.
- To study convergence, you typically:
  - Solve on a mesh with `nx = ny = 8`, then 16, then 32, etc.
  - Track quantities like max displacement or peak stress.
  - Check that they approach a stable value or analytical solution.

With scikit-fem, you can call `m.refined()` to perform uniform mesh refinement,
or regenerate the tensor mesh with higher `nx`, `ny`. You can also switch to
higher-order elements (e.g. P2 vs P1) to study p-refinement.

For quick experiments:

- Start with `nx = ny = 8` or 16 for speed.
- Increase gradually and watch runtime and memory usage.
- Plot convergence of a few key scalar metrics.

## 9. Notes on 3D, Nonlinear Materials, and Dynamic Problems

This notebook focused on **2D linear elasticity** for simplicity. scikit-fem can
also be used for:

- **3D problems**: volumetric meshes for solids (tetrahedral/hexahedral elements).
- **Nonlinear materials**: via custom weak forms and iterative solvers.
- **Other PDEs**: Poisson, diffusion, thermoelasticity, and more.
- **Time-dependent problems**: by combining spatial FEM with your own time-stepping
  loop in Python.

For trading/quant applications these features are less common, but they may
matter if you are interfacing with physical systems (cooling, structural
components, hardware enclosures, etc.) and want to prototype behavior in Python.

## 10. Best Practices and When to Use scikit-fem in Quant/Engineering Contexts

- **Keep meshes simple** for early experiments:
  - Coarse uniform meshes are fine for understanding trends and orders of magnitude.
  - Only refine once the basic setup (BCs, loads, materials) is clearly correct.
- **Validate against known solutions** when possible:
  - Compare uniaxial tension against analytical strain \(\varepsilon = \sigma / E\).
  - For simple beams/plates, compare deflections to textbook formulas.
- **Separate modeling from numerics**:
  - Encapsulate your scikit-fem setup (mesh + BCs + materials) in functions/classes.
  - Keep orchestration and post-processing in higher-level Python code.
- **Use Python for parameter sweeps**:
  - Vary material parameters, loads, or geometry and collect results.
  - Combine with NumPy/pandas to analyze sensitivities or risk scenarios.
- **Know when scikit-fem is appropriate**:
  - Prototyping **structural** or **physical** problems from Python, especially
    when you want more control than black-box solvers provide.
  - Embedding lightweight FEM checks inside larger quant/engineering pipelines.
  - For purely financial math (no physical PDEs), dedicated numerical libraries
    (NumPy/SciPy, PDE-specific solvers) are usually a better fit.

Used judiciously, scikit-fem gives you a **bridge between Python and FEM** so you
can explore physical models alongside your quantitative codebase without leaving
the Python ecosystem.

## 11. FEM Perspective on the Black–Scholes Equation

The **Black–Scholes PDE** for a European call option price $V(S, t)$ (no dividends,
constant volatility $\sigma$, risk-free rate $r$) is:

$$
\frac{\partial V}{\partial t}
+ \frac{1}{2} \sigma^2 S^2 \frac{\partial^2 V}{\partial S^2}
+ r S \frac{\partial V}{\partial S}
- r V = 0, \quad S > 0,\ 0 \le t < T,
$$

with terminal condition at maturity $t = T$:

$$
V(S, T) = \max(S - K, 0)
$$

and suitable boundary conditions as $S \to 0$ and $S \to \infty$ (or a truncated
finite interval for $S$).

### 11.1 Mapping Black–Scholes to a FEM-friendly form

FEM libraries like scikit-fem are built primarily for **elliptic** and **parabolic
PDEs** in **space** (and sometimes time), often in the form:

$$
- \nabla \cdot (a(x) \nabla u) + c(x) u = f(x)
$$

or its time-dependent variant. The Black–Scholes equation is **parabolic in time**
and **second order in space** (in the underlying price $S$ or transformed variable
$x = \ln S$).

A common approach to solve Black–Scholes with FEM:

1. Perform a **log-transformation**: $x = \ln S$, and possibly a time
   transformation to turn the PDE into a diffusion–reaction equation with
   more uniform coefficients.
2. Truncate the **spatial domain** to a finite interval $[x_{min}, x_{max}]$
   corresponding to $S \in [S_{min}, S_{max}]$.
3. Discretize **time** with a method like **implicit Euler** or
   **Crank–Nicolson**:

   - Replace $\partial V / \partial t$ by a finite difference between
     time levels $n$ and $n+1$.
   - At each time step, solve an **elliptic problem in space** for $V^{n+1}$.
4. Use FEM (e.g. with scikit-fem) to discretize the **spatial operator** in $x$
   at each time step.

Mathematically, at each time step you get a linear system of the form:

$$
(M + \Delta t \, A) V^{n+1} = M V^n + \Delta t \, F^n,
$$

where:

- $M$ is the **mass matrix** (from time discretization / reaction terms).
- $A$ is the **stiffness-like matrix** representing diffusion and drift terms
  in the spatial operator.
- $F^n$ collects any source terms and boundary conditions.

This is exactly the kind of system FEM is designed to handle.

### 11.2 Sketch of a 1D FEM Black–Scholes solver (conceptual)

A high-level workflow for a **1D FEM Black–Scholes** implementation (not full code):

1. **Define spatial domain** in $x = \ln S$:

   - Choose $S_{min}$, $S_{max}$; set $x_{min} = \ln S_{min}$, $x_{max} = \ln S_{max}$.
   - Create a 1D mesh over $[x_{min}, x_{max}]$ with nodes $x_i$.

2. **Assemble FEM matrices** (mass and stiffness) for the transformed PDE:

   - For each element, compute local mass and stiffness matrices.
   - Incorporate the coefficients corresponding to diffusion $\propto \sigma^2$
     and drift $\propto r$.

3. **Time stepping** from maturity $t = T$ backwards to $t = 0$:

   - Initialize $V^N$ at $t = T$ with payoff $\max(S - K, 0)$ at nodes.
   - For each step $n = N-1, \dots, 0$:
     - Apply boundary conditions (e.g. linear growth in $S$ as $S\to\infty$).
     - Solve $(M + \theta \, \Delta t \, A) V^{n+1} = (M - (1-\theta) \, \Delta t \, A) V^n$
       for $\theta = 1/2$ (Crank–Nicolson) or $1$ (implicit Euler).

4. **Interpolate** $V^0$ at your current $S_0$ (or $x_0 = \ln S_0$) for the option
   price at time 0.

### 11.3 How scikit-fem could be used here

scikit-fem is not a dedicated Black–Scholes solver, but you can reuse its **spatial
FEM machinery** if:

- You map your problem to a form supported by scikit-fem (e.g. a diffusion–reaction
  PDE in 1D or 2D) and
- You handle the **time discretization** in Python, calling scikit-fem at each time
  step to assemble/solve the spatial problem.

In the scikit-fem context, the workflow might be:

1. Use scikit-fem to define a **1D mesh** (or a thin 2D strip) representing $x$ or $S$.
2. Configure a problem type closest to the spatial operator in the transformed
   Black–Scholes PDE (e.g. a diffusion–reaction problem).
3. At each time step, update the right-hand side and solve for the new spatial
   field, retrieving node values as your $V^{n+1}$.

### 11.4 When FEM for Black–Scholes makes sense

FEM is especially attractive when:

- You have **complex spatial domains** (barriers, local volatility surfaces,
  state constraints) or
- You extend Black–Scholes to **multiple dimensions** (e.g. multi-asset options,
  stochastic volatility models) where standard finite-difference grids become
  cumbersome.

For vanilla, 1D Black–Scholes, **finite differences** or **semi-analytical
solutions** are usually simpler and faster. But FEM techniques—and tools like
scikit-fem—become powerful when you need to handle **PDEs on complex domains** or
with rich boundary/constraint structures, which occasionally appears in more
exotic option pricing and risk problems.